<a href="https://colab.research.google.com/github/brianchinphd/ProviderProfilingSimulation/blob/main/ProviderQualitySimulation_2023CMS_PartD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Provider Quality Simulation / Medication Adherence

This project uses the publicly available CMS dataset, *Medicare Part D Prescribers - by Provider and Drug (2023)*, retrieved from https://data.cms.gov/.

**Situation:** Because chronic medication adherence is a triple-weighted CMS Star Rating measure, I aimed to identify providers with low adherence for targeted intervention.

**Problem:** How can we identify primary care segments driving care gaps in chronic medication adherence (HEDIS/Star Rating risk)?

**Scope:** Hartford-based prescribers of ACE inhibitors (Lisinopril)

In [34]:
# Import Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Data
hfdmed = pd.read_csv('/content/Medicare_Part_D_Prescribers_by_Provider_and_Drug_2023.csv',low_memory=False)

# Generate New Dataframe
hfdace = hfdmed[hfdmed['Gnrc_Name'] == 'Lisinopril'].copy()
display(hfdace.head())

print(f"There were {hfdace.shape[0]} Hartford practioners who prescribed Lisinopril in 2023.")

,Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_Type,Prscrbr_Type_Src,Brnd_Name,Gnrc_Name,...,Tot_Day_Suply,Tot_Drug_Cst,Tot_Benes,GE65_Sprsn_Flag,GE65_Tot_Clms,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Tot_Day_Suply,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes
106,1003055021,Singh,Gagandeep,Hartford,CT,9,Hospitalist,Claim-Specialty,Lisinopril,Lisinopril,...,6521,3289.46,55.00,NaN,234.00,234.00,2726.79,5371.00,#,NaN
265,1003177965,Altszuler,David,Hartford,CT,9,Cardiology,Claim-Specialty,Lisinopril,Lisinopril,...,9059,930.13,33.00,#,NaN,NaN,NaN,NaN,#,NaN
305,1003270638,Poole,Nicole,Hartford,CT,9,Family Practice,Claim-Specialty,Lisinopril,Lisinopril,...,2980,252.02,12.00,NaN,35.00,100.00,252.02,2980.00,NaN,12.00
324,1003349960,Dhanabalsamy,Nisha,Hartford,CT,9,Internal Medicine,Claim-Specialty,Lisinopril,Lisinopril,...,1627,178.43,NaN,#,NaN,NaN,NaN,NaN,*,NaN
362,1003453184,Southwick,Brianna,Hartford,CT,9,Physician Assistant,Claim-Specialty,Lisinopril,Lisinopril,...,578,286.88,14.00,#,NaN,NaN,NaN,NaN,#,NaN


There were 306 Hartford practioners who prescribed Lisinopril in 2023.


I wanted to examine the 90-day fill ratios for Lisinopril, since this is a key indicator for HEDIS. This was not directly observable in this data, so I calculated the average days supply as a proxy measure to identify prescribers who may be driving adherence gaps.

In [35]:
# Calculate days-per-fill
hfdace['Days_Per_Fill'] = hfdace['Tot_Day_Suply'] / hfdace['Tot_Clms']
avg_days_per_fill = hfdace['Days_Per_Fill'].mean()
std_days_per_fill = hfdace['Days_Per_Fill'].std()
print(f"The average days per fill is: {avg_days_per_fill:.2f} days with SD of {std_days_per_fill:.2f} days")

The average days per fill is: 69.07 days with SD of 19.85 days


In [36]:
# Z-score the days-per-fill variable
hfdace['Days_Per_Fill_Zscore'] = (hfdace['Days_Per_Fill'] - avg_days_per_fill) / std_days_per_fill
pd.set_option('display.float_format', '{:.2f}'.format)

description_zscore = hfdace['Days_Per_Fill_Zscore'].describe()
display(description_zscore)

,Days_Per_Fill_Zscore
count,306.00
mean,0.00
std,1.00
min,-2.89
25%,-0.56
50%,0.37
75%,0.79
max,1.24


In [37]:
# Identify prescribers in the Bottom 2.5% of day-per-fill
hfdace['Days_Per_Fill_Flag'] = hfdace['Days_Per_Fill_Zscore'] < -2

# Count how many prescribers have a z-score < -2
num_low_adherence_prescribers = (hfdace['Days_Per_Fill_Zscore'] < -2).sum()
print(f"There were {num_low_adherence_prescribers} prescribers with z-score < 2")

There were 20 prescribers with z-score < 2


In [38]:
# Calculate mean claims and beneficiary volume
avg_clms = hfdace['Tot_Clms'].mean()
std_clms = hfdace['Tot_Clms'].std()
avg_benes = hfdace['Tot_Benes'].mean()
std_benes = hfdace['Tot_Benes'].std()
print(f"The average number of claims is: {avg_clms:.2f} with SD of {std_clms:.2f}")
print(f"The average number of benes is: {avg_benes:.2f} with SD of {std_benes:.2f}")

The average number of claims is: 98.39 with SD of 116.99
The average number of benes is: 41.46 with SD of 30.84


In [39]:
# Initialize low_adherence_prescribers with relevant columns, using .copy() to ensure it's an independent DataFrame
low_adherence_prescribers = hfdace[hfdace['Days_Per_Fill_Flag']][['Prscrbr_NPI', 'Tot_Clms', 'Tot_Benes']].copy()

# Define the conditions for the new Priority_Segment variable
conditions = [
    (low_adherence_prescribers['Tot_Clms'] < avg_clms) | (low_adherence_prescribers['Tot_Benes'].isna()),
    (low_adherence_prescribers['Tot_Clms'] > 215),
    (low_adherence_prescribers['Tot_Clms'] >= avg_clms) & (low_adherence_prescribers['Tot_Clms'] <= 215)]

# Define the corresponding choices for each condition
choices = ['Low Volume / Low Priority', 'High Priority', 'Medium Priority']

# Create the new 'Priority_Segment' column using numpy.select
low_adherence_prescribers['Priority_Segment'] = np.select(conditions, choices, default='Unknown')

# Display Prscrbr_NPI, Tot_Clms, Tot_Benes, and the new Priority_Segment for prescribers with Days_Per_Fill_Flag true
display(low_adherence_prescribers[['Prscrbr_NPI', 'Tot_Clms', 'Tot_Benes', 'Priority_Segment']])

,Prscrbr_NPI,Tot_Clms,Tot_Benes,Priority_Segment
106,1003055021,277,55.00,High Priority
362,1003453184,23,14.00,Low Volume / Low Priority
3852,1144227125,489,45.00,High Priority
4724,1174565253,14,NaN,Low Volume / Low Priority
5583,1194923128,601,85.00,High Priority
5797,1205085719,12,NaN,Low Volume / Low Priority
6507,1235205444,13,NaN,Low Volume / Low Priority
8369,1316043045,593,73.00,High Priority
9138,1336376706,11,NaN,Low Volume / Low Priority
9595,1366986119,13,NaN,Low Volume / Low Priority


**Summary**

* Out of 306 Hartford-based prescribers of ACE inhibitors, only a small segment (approx. 6%) represent high-conviction targets for adherence intervention.
* The "Low Adherence" designation corresponds to an Average Days Supply per Fill < 30 days (Z-score < 2), identifying providers whose prescribing habits structurally impact HEDIS/Star Rating performance.

**Actionable Plan**
* I recommend targeted outreach for the 7 High-Volume providers identified in the "High Priority" segment. These providers represent the greatest opportunity for targeted improvement to quality metrics.
* I recommend ongoing monitoring of the 13 Medium and Low Priority providers.
* No action is recommended for the remaining 286 providers.

**Limitations & Iteration**
* This project utilizes prescriber-level data; the next phase should integrate member-level data to validate whether these prescribing patterns result in actual Proportion of Days Covered gaps or if members are successfully utilizing split-fills.